In [10]:
import pandas as pd
import numpy as np
from pathlib import Path

def read_csv_safe(path):
    encodings = ["utf-8", "utf-8-sig", "gb18030", "gbk", "latin1"]
    last_error = None

    for enc in encodings:
        try:
            df = pd.read_csv(path, encoding=enc)
            print(f"[INFO] Read success: {path} | encoding={enc}")
            return df
        except Exception as e:
            last_error = e

    raise ValueError(f"Failed to read {path}. Last error: {last_error}")


# 1. 读取数据
events = read_csv_safe("C:/Users/31314/Desktop/手动匹配错误公司并删除后.csv")
prices = read_csv_safe("C:/Users/31314/Desktop/下载股价后.csv")

events["announcement_date_verified"] = pd.to_datetime(
    events["announcement_date_verified"], errors="coerce"
)
prices["date"] = pd.to_datetime(prices["date"], errors="coerce")

events["ticker"] = events["ticker"].astype(str).str.strip()
prices["ticker"] = prices["ticker"].astype(str).str.strip()

events = events.dropna(subset=["ticker", "announcement_date_verified"]).copy()
prices = prices.dropna(subset=["ticker", "date"]).copy()

# 2. 生成 event_id
events["event_date"] = events["announcement_date_verified"]
events["event_id"] = (
    events["ticker"].astype(str) + "_" +
    events["event_date"].dt.strftime("%Y-%m-%d")
)

event_cols = [
    "event_id", "company", "ticker", "event_date",
    "exchange", "ai_mentioned", "%_Laid_Off", "Funds_Raised_mm"
]
event_cols = [c for c in event_cols if c in events.columns]
events_small = events[event_cols].copy()

print("[INFO] events_small columns:", events_small.columns.tolist())

# 3. merge
panel = prices.merge(events_small, on="ticker", how="inner")
print("[INFO] panel columns after merge:", panel.columns.tolist())

panel["calendar_diff"] = (panel["date"] - panel["event_date"]).dt.days
panel = panel[(panel["calendar_diff"] >= -300) & (panel["calendar_diff"] <= 300)].copy()

print(f"[INFO] Rows after merge/filter: {len(panel)}")
print(f"[INFO] Unique events before t-definition: {panel['event_id'].nunique()}")


# 4. 定义相对交易日 t
def add_event_time(group):
    group = group.sort_values("date").reset_index(drop=True).copy()

    event_date = group["event_date"].iloc[0]
    candidate = np.where(group["date"] >= event_date)[0]

    if len(candidate) == 0:
        group["t"] = np.nan
        group["is_event_day"] = 0
        group["event_trading_date"] = pd.NaT
        return group

    event_pos = candidate[0]
    event_trading_date = group.loc[event_pos, "date"]

    group["t"] = np.arange(len(group)) - event_pos
    group["is_event_day"] = (group["t"] == 0).astype(int)
    group["event_trading_date"] = event_trading_date
    return group

panel = (
    panel.groupby("event_id", group_keys=False)
         .apply(add_event_time)
         .reset_index(drop=True)
)

print("[INFO] panel columns after groupby/apply:", panel.columns.tolist())

panel = panel[(panel["t"] >= -260) & (panel["t"] <= 60)].copy()

# 5. 保存
out_file = Path("C:/Users/31314/Desktop/定义每个事件的数据窗口.csv")
out_file.parent.mkdir(parents=True, exist_ok=True)
panel.to_csv(out_file, index=False)

print(f"[INFO] Saved to: {out_file}")
print(f"[INFO] Final rows: {len(panel)}")
print(f"[INFO] Final unique events: {panel['event_id'].nunique()}")

[INFO] Read success: layoffs.csv | encoding=gb18030
[INFO] Read success: data/processed/price_panel_raw.csv | encoding=utf-8
[INFO] events_small columns: ['event_id', 'company', 'ticker', 'event_date', 'exchange', 'ai_mentioned']
[INFO] panel columns after merge: ['date', 'ticker', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'adj_close', 'Volume', 'ret', 'event_id', 'company', 'event_date', 'exchange', 'ai_mentioned']
[INFO] Rows after merge/filter: 254925
[INFO] Unique events before t-definition: 639


/tmp/ipykernel_21179/1849172190.py:85: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(add_event_time)


[INFO] panel columns after groupby/apply: ['date', 'ticker', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'adj_close', 'Volume', 'ret', 'event_id', 'company', 'event_date', 'exchange', 'ai_mentioned', 'calendar_diff', 't', 'is_event_day', 'event_trading_date']
[INFO] Saved to: data/processed/returns_panel_step0.csv
[INFO] Final rows: 167079
[INFO] Final unique events: 638
